In [0]:
%run ../config/config

In [0]:
# Imports
import logging
import socket
import os
import sys
import time

sys.path.insert(0, str(project_path))
from utils.logger import MLOpsLogger, log_dataframe_info
import pandas as pd

In [0]:
# Logger inference
logger = MLOpsLogger(
    name='05_inference',
    log_level='DEBUG' if DEBUG_MLOPS else 'INFO',
    log_dir=logs_path,
    is_job_run=True,
    job_context={
        'mes_vta': mes_vta,
        'environment': env,
        'notebook': '05_inference'
    }
)

In [0]:
# =============================================================================
# PARÁMETROS DE EJECUCIÓN MULTI-MES (desde config centralizado)
# =============================================================================

from utils.month_delta import month_delta
RUN_HISTORICAL = first_run  # Controlado por bandera FIRST_RUN del config
if RUN_HISTORICAL and num_meses_historicos > 0:
    # Generar lista de meses HISTÓRICOS (sin incluir el actual)
    meses_historicos_lista = []
    mes_tmp = mes_vta

    for i in range(num_meses_historicos):
        mes_tmp = month_delta(str(mes_tmp), -1)
        meses_historicos_lista.append(mes_tmp)

    # Invertir para procesar desde el más antiguo al más reciente
    meses_historicos_lista.reverse()

    # Agregar el mes actual al final
    meses_a_procesar = meses_historicos_lista + [mes_vta]

    logger.info("="*80)
    logger.info("MODO HISTÓRICO ACTIVADO (FIRST_RUN)")
    logger.info("="*80)
    logger.info(f" Se procesarán {len(meses_a_procesar)} meses para inferencia:")
    logger.info(f"  Históricos: {meses_historicos_lista}")
    logger.info(f"  Actual: {mes_vta}")
    logger.info("="*80)
else:
    meses_a_procesar = [mes_vta]
    logger.info(f"Procesando únicamente el mes actual: {mes_vta}")

In [0]:
import sasindb
from sasindb.mla import Model
from pyspark.sql import DataFrame, SQLContext, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import lit
from pyspark.sql.window import Window
import time

# =============================================================================
# LOOP PRINCIPAL POR CADA MES
# =============================================================================
resultados_inferencia = []
for idx_mes, mes_actual_proceso in enumerate(meses_a_procesar, 1):
    logger.info("")
    logger.info("="*80)
    logger.info(f"PROCESANDO MES {idx_mes}/{len(meses_a_procesar)}: {mes_actual_proceso}")
    logger.info("="*80)

    full_table_name_mt = TABLE_MASTER
    try:
        if use_segments:
            # MODO CON SEGMENTOS
            logger.info("Leyendo master table...")
            df_master = spark.sql(f"SELECT * FROM {full_table_name_mt} WHERE codmes = {mes_actual_proceso}")
            total_records = df_master.count()
            logger.info(f"Total de registros en master table: {total_records:,}")

            scored_dataframes = []

            for seg_config in segments_config:
                segment_id = seg_config["segment_id"]
                segment_name = seg_config["name"]
                segment_filter = seg_config["filter"]
                model_name = seg_config["model_table"]

                logger.info(f"PROCESANDO SEGMENTO {segment_id}: {segment_name}")

                df_segment = df_master.filter(segment_filter).withColumn("segment_id", lit(segment_id))
                count_segment = df_segment.count()
                logger.info(f"Registros en segmento {segment_id}: {count_segment:,}")

                # Guardar segmento
                segment_table = f"{catalog_name}.{schema_name}.bcp_segmento_{segment_id}_{mes_actual_proceso}"
                df_segment.write.mode("overwrite").saveAsTable(segment_table)

                # Aplicar modelo
                model_table = f"{catalog_name}.{SCHEMA_MODELS}.{model_name}"
                modelds = spark.table(model_table)
                batch_model = Model.create(modelds, df_segment)
                batch_model.setDBMaxText(32767)
                scored = batch_model.score()

                # Guardar scoring temporal
                scored_table = f"{catalog_name}.{schema_name}.bcp_scored_seg{segment_id}_{mes_actual_proceso}"
                scored.write.mode("overwrite").saveAsTable(scored_table)

                scored_dataframes.append(scored)

            # Consolidar segmentos
            df_final = scored_dataframes[0]
            for df in scored_dataframes[1:]:
                df_final = df_final.union(df)

        else:
            # MODO SIN SEGMENTOS
            df_master = spark.sql(f"SELECT * FROM {full_table_name_mt} WHERE codmes = {mes_actual_proceso}")
            total_records = df_master.count()
            model_name = TABLE_MODEL
            model_table = f"{catalog_name}.{SCHEMA_MODELS}.{model_name}"
            modelds = spark.table(model_table)
            batch_model = Model.create(modelds, df_master)
            batch_model.setDBMaxText(32767)
            df_final = batch_model.score()

        # FIX - El scoreo altera la tabla de input: Mantener solo las columnas generadas por .score()
        df_final = df_final.select('codclavepartycli', 'P_def30_121') # codmes no es necesario, el loop recorre 1 a 1
        df_final = df_master.join(df_final, 'codclavepartycli', 'left')

        # numpd, numxb, numsc son nombres necesarios para la implementación
        df_final = df_final.withColumnRenamed('P_def30_121', 'numpd') # 'P_def30_121' puede parametrizarse en config.ipynb
        df_final = df_final.withColumn("numxb", F.log(F.col("numpd") / (1 - F.col("numpd"))))
        df_final = df_final.withColumn(
            "numsc",
            F.when(
                F.col("numxb").isNotNull(),
                F.greatest( F.lit(3), F.round(174.25 - 57.708 * F.col("numxb"), 1) )
            )
        )

        # SIEMPRE: Eliminar mes existente antes de insertar (evitar duplicados)
        logger.info(f"Sobrescribiendo datos existentes para codmes={mes_actual_proceso} usando replaceWhere")

        # Verificar si la tabla existe (para logging consistente)
        try:
            # Safer way to check for table existence with 3-part namespace
            spark.read.table(output_table).limit(0)
            table_exists = True
        except:
            table_exists = False

        if table_exists:
            logger.info(f"Tabla {output_table} existe. Usando replaceWhere para mes {mes_actual_proceso}...")
            # Usar overwrite con replaceWhere para asegurar atomicidad
            df_final.write \
                .format("delta") \
                .mode("overwrite") \
                .option("replaceWhere", f"codmes = {mes_actual_proceso}") \
                .saveAsTable(output_table)
            logger.info("Datos sobrescritos exitosamente")
        else:
            # Crear tabla por primera vez
            logger.info(f"Creando tabla {output_table} por primera vez...")
            df_final.write.mode("overwrite").format("delta").saveAsTable(output_table)

        final_count = df_final.count()
        logger.info(f"✓ Inferencia completada para mes {mes_actual_proceso}: {final_count:,} registros")

        resultados_inferencia.append({
            'mes': mes_actual_proceso,
            'registros': final_count,
            'exitoso': True
        })

    except Exception as e:
        logger.error(f"Error procesando mes {mes_actual_proceso}: {str(e)}")
        resultados_inferencia.append({
            'mes': mes_actual_proceso,
            'registros': 0,
            'exitoso': False,
            'error': str(e)
        })
        continue

# =============================================================================
# RESUMEN FINAL
# =============================================================================
logger.info("")
logger.info("="*80)
logger.info("RESUMEN FINAL DE INFERENCIA")
logger.info("="*80)
exitosos = sum(1 for r in resultados_inferencia if r['exitoso'])
logger.info(f"Total meses procesados: {len(resultados_inferencia)}")
logger.info(f"Exitosos: {exitosos}")
logger.info(f"Fallidos: {len(resultados_inferencia) - exitosos}")
logger.info("="*80)